# DDColor 古写真ファインチューニング（SHIMA CRAFT カラー化サービス用）

ブラウザで動かしている高品質カラー化モデル **DDColor-T** を、
「経年劣化した日本の白黒写真」向けに追加学習させるノートブックです。

現代のカラー写真を人工的に劣化（粒状ノイズ・退色・傷・周辺減光・低画質化）させて
「古い白黒写真 → 正しい色」の学習ペアを合成し、モデルを再学習します。

## 必要なもの
- Google アカウント（Colab 無料枠。**クレジットカード不要**）
- Google Drive の空き容量 約2GB（チェックポイント・出力の保存用）

## 使い方
1. メニュー「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選択
2. 上から順にセルを実行（▶ボタン）
3. 学習は `QUICK` 設定で約1〜2時間。Colab が切断されても再実行すれば
   Drive のチェックポイントから自動再開します
4. 最後のセルが `ddcolor_finetuned.zip` を Drive に保存します →
   中のファイルをリポジトリの `public/models/` に上書きして差し替え完了

## 手順の流れ
| セル | 内容 | 目安時間 |
|---|---|---|
| 1. セットアップ | ライブラリ導入・DDColor取得・Drive接続 | 3分 |
| 2. 設定 | データ量・学習ステップ数の選択 | - |
| 3. 劣化シミュレーション | 学習データ合成の確認 | 1分 |
| 4. データ準備 | COCO画像のダウンロードと前処理 | 10〜30分 |
| 5. 学習 | ファインチューニング本体 | 1〜4時間 |
| 6. 評価 | 元モデルとの比較（手持ちの古写真でも可） | 5分 |
| 7. ONNX出力 | ブラウザ用モデルファイル生成 | 10分 |


In [ ]:
# @title 1. セットアップ（ライブラリ・DDColor・Google Drive）
import subprocess, sys, os

# GPU 確認
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("GPUが有効ではありません。「ランタイム」→「ランタイムのタイプを変更」→ T4 GPU を選んでください")
print("GPU:", gpu.stdout.strip())

# 依存ライブラリ（Colab には torch/torchvision 導入済み）
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm", "opencv-python-headless", "huggingface_hub",
                "onnx", "onnxruntime", "onnxconverter-common", "onnxslim",
                "onnxscript", "tqdm"], check=True)  # onnxscript: 新しいtorchのONNX出力に必要

# DDColor 公式リポジトリ（Apache-2.0）
if not os.path.exists("DDColor"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/piddnad/DDColor.git"], check=True)
sys.path.insert(0, "DDColor")

# Google Drive（チェックポイント・出力の保存先）
from google.colab import drive
drive.mount("/content/drive")
WORK_DIR = "/content/drive/MyDrive/ddcolor_finetune"
os.makedirs(WORK_DIR, exist_ok=True)
print("作業ディレクトリ:", WORK_DIR)


In [ ]:
# @title 2. 設定
CONFIG = {
    # "quick": COCO val2017 (5,000枚・1GB) で1〜2時間の試行。まずはこちら
    # "full" : COCO train2017 の人物画像 25,000枚で本格学習（4時間×数セッション）
    "dataset": "quick",  # @param ["quick", "full"]

    # 学習ステップ数。quick なら 6000、full なら 20000 以上を推奨
    "train_steps": 6000,  # @param {type:"integer"}

    "batch_size": 12,          # T4 (16GB) 向け。メモリ不足なら 8 に下げる
    "crop_size": 256,          # DDColor の学習解像度（推論は512のまま）
    "lr_encoder": 5e-6,        # エンコーダは小さく（既習知識の破壊を防ぐ）
    "lr_decoder": 2e-5,        # デコーダ・精緻化層は大きめ
    "weight_decay": 1e-4,
    "chroma_loss_weight": 0.2, # 彩度不足（セピア化）へのペナルティ
    "checkpoint_every": 500,   # Drive への保存間隔（切断対策）
    "val_count": 24,           # 検証用に取り分ける枚数
    "seed": 42,
}
print(CONFIG)


In [ ]:
# @title 古写真劣化シミュレーション（学習データ合成の核）
# 現代のカラー写真から「昭和の白黒プリントをスキャンしたような」入力画像を合成する。
# 学習ペア = (劣化白黒, 元のカラーの ab チャンネル)
import numpy as np
import cv2


def _radial_vignette(h, w, strength, rng):
    cy = h * (0.5 + rng.uniform(-0.1, 0.1))
    cx = w * (0.5 + rng.uniform(-0.1, 0.1))
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    r = r / (np.sqrt(cy**2 + cx**2) + 1e-6)
    return 1.0 - strength * np.clip(r, 0, 1) ** 2


def _add_scratches(gray, rng):
    h, w = gray.shape
    n = rng.integers(1, 4)
    canvas = (gray * 255).astype(np.uint8).copy()
    for _ in range(n):
        x0, y0 = rng.integers(0, w), rng.integers(0, h)
        x1, y1 = x0 + rng.integers(-w // 3, w // 3), y0 + rng.integers(-h // 3, h // 3)
        color = int(rng.choice([230, 245, 30]))
        cv2.line(canvas, (x0, y0), (int(x1), int(y1)), color, 1, cv2.LINE_AA)
    return canvas.astype(np.float32) / 255.0


def _halftone(gray, rng):
    """網点印刷（書籍・新聞に載った写真のスキャン）を模す。
    回転したドットグリッド上で、暗いほど大きい点を打つ。"""
    h, w = gray.shape
    cell = float(rng.uniform(2.5, 6.0))       # 網点ピッチ(px)
    angle = rng.uniform(0, np.pi)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    u = xx * np.cos(angle) + yy * np.sin(angle)
    v = -xx * np.sin(angle) + yy * np.cos(angle)
    du = u / cell - np.floor(u / cell + 0.5)
    dv = v / cell - np.floor(v / cell + 0.5)
    r = np.sqrt(du**2 + dv**2)                # セル中心からの距離 0〜0.7
    dot_r = np.sqrt(np.clip(1.0 - gray, 0, 1)) * 0.58
    ht = 1.0 - (r < dot_r).astype(np.float32)
    ht = cv2.GaussianBlur(ht, (0, 0), sigmaX=max(0.5, cell * 0.3))
    k = rng.uniform(0.5, 1.0)                 # 完全網点〜部分的
    return gray * (1 - k) + ht * k


def _press_contrast(gray, rng):
    """報道・印刷写真特有の強コントラスト（黒潰れ・白飛び）を模すS字カーブ。"""
    gain = rng.uniform(1.3, 2.2)
    return np.clip((gray - 0.5) * gain + 0.5, 0, 1)


def _add_dust(gray, rng):
    h, w = gray.shape
    n = rng.integers(5, 40)
    canvas = (gray * 255).astype(np.uint8).copy()
    for _ in range(n):
        x, y = rng.integers(0, w), rng.integers(0, h)
        r = int(rng.integers(1, 3))
        color = int(rng.choice([240, 250, 20]))
        cv2.circle(canvas, (x, y), r, color, -1, cv2.LINE_AA)
    return canvas.astype(np.float32) / 255.0


def degrade_to_old_bw(img_rgb, rng=None):
    """カラー画像(float32 RGB 0..1)を古い白黒写真風に劣化させ、グレーRGB(0..1)で返す。"""
    if rng is None:
        rng = np.random.default_rng()
    h, w = img_rgb.shape[:2]

    # 1) 白黒化: 当時のフィルム特性を模してチャンネル重みをランダム化
    #    （オルソフィルムは赤に鈍感 → 唇・肌が暗く写る等の再現）
    wr = rng.uniform(0.05, 0.45)
    wg = rng.uniform(0.4, 0.65)
    wb = max(0.0, 1.0 - wr - wg)
    gray = img_rgb[..., 0] * wr + img_rgb[..., 1] * wg + img_rgb[..., 2] * wb

    # 2) 退色: 黒の浮き・白の沈み（トーンレンジ圧縮）
    lo = rng.uniform(0.0, 0.18)
    hi = rng.uniform(0.75, 1.0)
    gray = lo + gray * (hi - lo)

    # 3) ガンマゆらぎ（現像・スキャンのばらつき）
    gray = np.clip(gray, 0, 1) ** rng.uniform(0.75, 1.3)

    # 3b) 報道・印刷写真の強コントラスト（黒潰れ・白飛び）
    if rng.random() < 0.3:
        gray = _press_contrast(gray, rng)

    # 4) 局所コントラスト低下（にじみ・経年のカブリ）
    k = rng.uniform(0.0, 0.35)
    if k > 0.01:
        blur = cv2.GaussianBlur(gray, (0, 0), sigmaX=max(1.5, w / 80))
        gray = gray * (1 - k) + blur * k

    # 5) ピンぼけ
    sigma = rng.uniform(0.0, 1.2)
    if sigma > 0.05:
        gray = cv2.GaussianBlur(gray, (0, 0), sigmaX=sigma)

    # 6) フィルム粒状ノイズ（時々、粗い粒子）
    noise_sigma = rng.uniform(0.005, 0.04)
    noise = rng.normal(0, noise_sigma, gray.shape).astype(np.float32)
    if rng.random() < 0.4:
        noise = cv2.GaussianBlur(noise, (0, 0), sigmaX=rng.uniform(0.6, 1.2))
        noise *= 2.0
    gray = gray + noise

    # 7) 周辺減光
    if rng.random() < 0.5:
        gray = gray * _radial_vignette(h, w, rng.uniform(0.05, 0.25), rng)

    gray = np.clip(gray, 0, 1)

    # 7b) 網点印刷（書籍・新聞掲載写真のスキャン。硬い難例なのでやや高確率）
    if rng.random() < 0.3:
        gray = np.clip(_halftone(gray, rng), 0, 1)

    # 8) 傷・ホコリ（低確率）
    if rng.random() < 0.25:
        gray = _add_scratches(gray, rng)
    if rng.random() < 0.3:
        gray = _add_dust(gray, rng)

    # 9) JPEG/印刷アーティファクト
    if rng.random() < 0.6:
        q = int(rng.integers(40, 90))
        ok, enc = cv2.imencode(".jpg", (gray * 255).astype(np.uint8),
                               [cv2.IMWRITE_JPEG_QUALITY, q])
        if ok:
            gray = cv2.imdecode(enc, cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0

    return np.repeat(gray[..., None], 3, axis=2)


def rgb_to_lab_ab(img_rgb):
    """sRGB(float32 0..1) → CIE Lab の (L, ab)。L:0..100 / ab:実スケール(±127)。"""
    lab = cv2.cvtColor(img_rgb.astype(np.float32), cv2.COLOR_RGB2Lab)
    return lab[..., 0], lab[..., 1:]


In [ ]:
# @title 劣化サンプルの確認（COCOのサンプル1枚で3パターン生成）
import os
import urllib.request
import numpy as np
import cv2
import matplotlib.pyplot as plt

sample_path = "/content/sample.jpg"
if not os.path.exists(sample_path):
    urllib.request.urlretrieve(
        "http://images.cocodataset.org/val2017/000000039769.jpg", sample_path)

img = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(img); axes[0].set_title("original")
for i in range(3):
    axes[i + 1].imshow(degrade_to_old_bw(img, rng))
    axes[i + 1].set_title(f"degraded {i + 1}")
for ax in axes: ax.axis("off")
plt.show()


In [ ]:
# @title 4. データ準備（ダウンロード → 512px前処理 → Dataset）
import os
import sys
import subprocess
import glob
import zipfile
import urllib.request
import numpy as np
import cv2
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader

RAW_DIR = "/content/raw_images"
PREP_DIR = "/content/prep_images"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PREP_DIR, exist_ok=True)

# ---- ダウンロード ----
if CONFIG["dataset"] == "quick":
    zip_path = "/content/val2017.zip"
    if not glob.glob(f"{RAW_DIR}/*.jpg"):
        print("COCO val2017 (約1GB) をダウンロード中…")
        urllib.request.urlretrieve("http://images.cocodataset.org/zips/val2017.zip", zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall("/content")
        os.rename("/content/val2017", RAW_DIR + "_tmp")
        os.rmdir(RAW_DIR)
        os.rename(RAW_DIR + "_tmp", RAW_DIR)
        os.remove(zip_path)
else:
    # full: fiftyone で人物を含む画像だけを選択ダウンロード
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fiftyone"], check=True)
    import fiftyone.zoo as foz
    ds = foz.load_zoo_dataset(
        "coco-2017", split="train", label_types=["detections"],
        classes=["person"], max_samples=25000, only_matching=True,
        dataset_dir="/content/fiftyone_coco",
    )
    for s in ds:
        dst = os.path.join(RAW_DIR, os.path.basename(s.filepath))
        if not os.path.exists(dst):
            os.link(s.filepath, dst)

# ※ 自分で集めた現代のカラー写真（家族写真・祭り・集合写真など）を混ぜると
#   さらにドメインが近づきます。Drive に置いて下の行のコメントを外してください:
# RAW_EXTRA = "/content/drive/MyDrive/ddcolor_finetune/my_photos"  # 追加写真フォルダ

# ---- 前処理: 長辺512pxに縮小して保存（毎エポックのリサイズを省く） ----
raw_files = sorted(glob.glob(f"{RAW_DIR}/*.jpg"))
print(f"元画像: {len(raw_files)}枚")
if not glob.glob(f"{PREP_DIR}/*.jpg"):
    for p in tqdm(raw_files, desc="前処理"):
        img = cv2.imread(p)
        if img is None or min(img.shape[:2]) < 260:
            continue  # 小さすぎる画像は除外
        h, w = img.shape[:2]
        scale = 512 / max(h, w)
        if scale < 1:
            img = cv2.resize(img, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)
        cv2.imwrite(os.path.join(PREP_DIR, os.path.basename(p)), img,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])

all_files = sorted(glob.glob(f"{PREP_DIR}/*.jpg"))
val_files = all_files[: CONFIG["val_count"]]
train_files = all_files[CONFIG["val_count"]:]
print(f"学習: {len(train_files)}枚 / 検証: {len(val_files)}枚")


class OldPhotoDataset(Dataset):
    """(劣化白黒グレーRGB, 元画像のab) の学習ペアを返す。"""

    def __init__(self, files, crop):
        self.files = files
        self.crop = crop

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        rng = np.random.default_rng()
        img = cv2.imread(self.files[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        h, w = img.shape[:2]
        c = self.crop
        # ランダムスケール＋クロップ
        scale = rng.uniform(1.0, 1.5) * c / min(h, w)
        if scale < 1 or scale > 1:
            img = cv2.resize(img, (max(c, round(w * scale)), max(c, round(h * scale))),
                             interpolation=cv2.INTER_AREA)
        h, w = img.shape[:2]
        y0 = rng.integers(0, h - c + 1)
        x0 = rng.integers(0, w - c + 1)
        img = img[y0:y0 + c, x0:x0 + c]
        if rng.random() < 0.5:
            img = img[:, ::-1].copy()  # 左右反転

        degraded = degrade_to_old_bw(img, rng)          # 入力（グレーRGB 0..1）
        _, ab = rgb_to_lab_ab(img)                       # 目標（実スケール ab）
        x = torch.from_numpy(degraded.transpose(2, 0, 1))
        y = torch.from_numpy(ab.transpose(2, 0, 1))
        return x, y


train_ds = OldPhotoDataset(train_files, CONFIG["crop_size"])
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
print("DataLoader 準備完了")


In [ ]:
# @title 5. ファインチューニング本体（切断されても再実行で自動再開）
import os
import json
import numpy as np
import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from basicsr.archs.ddcolor_arch import DDColor

device = "cuda"
torch.manual_seed(CONFIG["seed"])


def build_ddcolor_tiny():
    """HF の設定・重みから DDColor-T を構築する（ローカル検証済みの手順）。"""
    cfg = json.load(open(hf_hub_download("piddnad/ddcolor_paper_tiny", "config.json")))
    model = DDColor(
        encoder_name=cfg["encoder_name"],
        decoder_name=cfg["decoder_name"],
        input_size=cfg["input_size"],
        num_output_channels=cfg["num_output_channels"],
        last_norm=cfg["last_norm"],
        do_normalize=cfg["do_normalize"],
        num_queries=cfg["num_queries"],
        num_scales=cfg["num_scales"],
        dec_layers=cfg["dec_layers"],
    )
    state = torch.load(hf_hub_download("piddnad/ddcolor_paper_tiny", "pytorch_model.bin"),
                       map_location="cpu")
    if "params" in state:
        state = state["params"]
    missing, unexpected = model.load_state_dict(state, strict=False)
    assert not missing and not unexpected, f"重み不一致: missing={missing} unexpected={unexpected}"
    return model


model = build_ddcolor_tiny().to(device)

# パラメータグループ: エンコーダは弱く、デコーダ系は強めに学習
enc_params, dec_params = [], []
for name, p in model.named_parameters():
    (enc_params if name.startswith("encoder") else dec_params).append(p)
optimizer = torch.optim.AdamW(
    [{"params": enc_params, "lr": CONFIG["lr_encoder"]},
     {"params": dec_params, "lr": CONFIG["lr_decoder"]}],
    weight_decay=CONFIG["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["train_steps"])
scaler = torch.cuda.amp.GradScaler()

# ---- チェックポイントからの自動再開 ----
ckpt_path = os.path.join(WORK_DIR, "ckpt_latest.pt")
start_step = 0
if os.path.exists(ckpt_path):
    ck = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    scaler.load_state_dict(ck["scaler"])
    start_step = ck["step"]
    print(f"チェックポイントから再開: step {start_step}")

if start_step >= CONFIG["train_steps"]:
    print("⚠ 既に指定ステップ数まで学習済みのため、このままでは1歩も学習しません。")
    print("  続きから学習するには、設定セルの train_steps を増やして（例: 12000）、")
    print("  設定セル → このセルの順に再実行してください。")


def chroma_deficit_loss(pred_ab, gt_ab):
    """出力の彩度が正解より低い分だけペナルティ（セピア化・灰色化を防ぐ）。"""
    c_pred = torch.sqrt(pred_ab[:, 0] ** 2 + pred_ab[:, 1] ** 2 + 1e-6)
    c_gt = torch.sqrt(gt_ab[:, 0] ** 2 + gt_ab[:, 1] ** 2 + 1e-6)
    return F.relu(c_gt - c_pred).mean() / 110.0


model.train()
step = start_step
losses = []
data_iter = iter(train_loader)
from tqdm import tqdm
pbar = tqdm(total=CONFIG["train_steps"], initial=start_step, desc="学習")
while step < CONFIG["train_steps"]:
    try:
        x, y = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        x, y = next(data_iter)
    x = x.to(device, non_blocking=True)
    y = y.to(device, non_blocking=True)

    optimizer.zero_grad(set_to_none=True)
    with torch.cuda.amp.autocast():
        pred = model(x)
        loss_l1 = F.l1_loss(pred, y) / 110.0
        loss_ch = chroma_deficit_loss(pred, y)
        loss = loss_l1 + CONFIG["chroma_loss_weight"] * loss_ch
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    losses.append(float(loss))
    step += 1
    pbar.update(1)
    if step % 50 == 0:
        pbar.set_postfix(loss=f"{np.mean(losses[-50:]):.4f}")
    if step % CONFIG["checkpoint_every"] == 0 or step == CONFIG["train_steps"]:
        torch.save({"step": step, "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "scaler": scaler.state_dict()}, ckpt_path)
pbar.close()

if losses:
    import matplotlib.pyplot as plt
    win = min(50, len(losses))
    plt.figure(figsize=(8, 3))
    plt.plot(np.convolve(losses, np.ones(win) / win, mode="valid"))
    plt.title("loss (移動平均)"); plt.xlabel("step"); plt.grid(True)
    plt.show()
    print("学習完了。チェックポイント:", ckpt_path)


In [ ]:
# @title 6. 評価 — 元モデルとの比較（検証画像＋手持ちの古写真）
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt


def colorize_512(m, gray_rgb01):
    """グレーRGB(0..1, HxWx3) → カラーRGB(0..1)。ブラウザと同じ 512 推論 + 元解像度L合成。"""
    h, w = gray_rgb01.shape[:2]
    inp = cv2.resize(gray_rgb01, (512, 512), interpolation=cv2.INTER_AREA)
    x = torch.from_numpy(inp.transpose(2, 0, 1))[None].to(device)
    with torch.no_grad(), torch.cuda.amp.autocast():
        ab512 = m(x)[0].float().cpu().numpy().transpose(1, 2, 0)
    ab = cv2.resize(ab512, (w, h), interpolation=cv2.INTER_LINEAR)
    L = cv2.cvtColor(gray_rgb01.astype(np.float32), cv2.COLOR_RGB2Lab)[..., 0]
    lab = np.dstack([L, ab]).astype(np.float32)
    return np.clip(cv2.cvtColor(lab, cv2.COLOR_Lab2RGB), 0, 1)


baseline = build_ddcolor_tiny().to(device).eval()
model.eval()

# ---- 検証画像（劣化合成）で比較: 入力 / 元モデル / 学習後 / 正解 ----
rng = np.random.default_rng(7)
n_show = min(6, len(val_files))
fig, axes = plt.subplots(n_show, 4, figsize=(18, 4.5 * n_show))
for i in range(n_show):
    img = cv2.cvtColor(cv2.imread(val_files[i]), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    degraded = degrade_to_old_bw(img, rng)
    for j, (title, im) in enumerate([
        ("input (劣化白黒)", degraded),
        ("元モデル", colorize_512(baseline, degraded)),
        ("ファインチューニング後", colorize_512(model, degraded)),
        ("正解", img),
    ]):
        axes[i, j].imshow(im)
        axes[i, j].set_title(title if i == 0 else "")
        axes[i, j].axis("off")
plt.tight_layout(); plt.show()

# ---- 手持ちの本物の古写真でも比較（任意・複数可） ----
print("本物の古い白黒写真があればアップロードしてください（スキップ可: キャンセルを押す）")
from google.colab import files as colab_files
try:
    uploaded = colab_files.upload()
except Exception:
    uploaded = {}
for name in uploaded:
    raw = cv2.imdecode(np.frombuffer(uploaded[name], np.uint8), cv2.IMREAD_COLOR)
    g = cv2.cvtColor(raw, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    gray_rgb = np.repeat(g[..., None], 3, axis=2)
    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    for j, (title, im) in enumerate([
        (name, gray_rgb),
        ("元モデル", colorize_512(baseline, gray_rgb)),
        ("ファインチューニング後", colorize_512(model, gray_rgb)),
    ]):
        axes[j].imshow(im); axes[j].set_title(title); axes[j].axis("off")
    plt.tight_layout(); plt.show()


In [ ]:
# @title 7. ONNX出力 — ブラウザ用モデルファイル生成（fp32=WebGPU / fp16=WASM）
import warnings
warnings.filterwarnings("ignore")  # fp16変換の桁落ち警告が大量に出るため抑止
import os
import json
import hashlib
import shutil
import numpy as np
import torch
import onnx
from onnxslim import slim
from onnxconverter_common import float16

EXPORT_DIR = "/content/export"
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
os.makedirs(EXPORT_DIR, exist_ok=True)
PART_SIZE = 90 * 1024 * 1024  # 90MiB（GitHubの100MB制限回避・現行と同じ）

model.eval().cpu()
dummy = torch.rand(1, 3, 512, 512)
dummy = dummy[:, :1].repeat(1, 3, 1, 1)

raw_path = os.path.join(EXPORT_DIR, "raw_export.onnx")
torch.onnx.export(
    model, dummy, raw_path,
    input_names=["input"], output_names=["output"],
    opset_version=17, do_constant_folding=True,
)

# 新しいPyTorchは重みを外部データ(.data)に分離して出力することがあるため、
# いったん読み込んで単一ファイルへ内部化する（分離のままだと配信ファイルが壊れる）
m = onnx.load(raw_path)  # 同ディレクトリの外部データを自動解決
fp32_path = os.path.join(EXPORT_DIR, "ddcolor_finetuned_fp32.onnx")
onnx.save_model(m, fp32_path, save_as_external_data=False)
for f in os.listdir(EXPORT_DIR):  # raw と外部データの掃除
    if f.startswith("raw_export"):
        os.remove(os.path.join(EXPORT_DIR, f))

# onnxslim で簡約（新exporterが残す no-op Cast の除去。fp16変換の失敗も防ぐ）
slim(fp32_path, fp32_path)

# 数値検証: PyTorch と ONNX の出力一致を確認
import onnxruntime as ort
sess = ort.InferenceSession(fp32_path)
with torch.no_grad():
    ref = model(dummy).numpy()
got = sess.run(None, {"input": dummy.numpy()})[0]
diff = float(np.abs(got - ref).max())
print(f"ONNX検証(fp32): 最大誤差 {diff:.4f}（0.1未満ならOK）")
assert diff < 0.1

# fp16 版（WASM用。入出力はfp32のまま重みのみfp16化 — 現行構成と同じ）
fp16_path = os.path.join(EXPORT_DIR, "ddcolor_finetuned_fp16.onnx")
m16 = float16.convert_float_to_float16(onnx.load(fp32_path), keep_io_types=True)
onnx.save(m16, fp16_path)

# fp16 も読み込み・数値検証してから配信ファイルにする
got16 = ort.InferenceSession(fp16_path).run(None, {"input": dummy.numpy()})[0]
diff16 = float(np.abs(got16 - ref).mean())
print(f"ONNX検証(fp16): 平均誤差 {diff16:.4f}（0.5未満ならOK）")
assert diff16 < 0.5


def split_parts(src, base_name):
    data = open(src, "rb").read()
    parts = []
    for i in range(0, len(data), PART_SIZE):
        chunk = data[i:i + PART_SIZE]
        name = f"{base_name}.onnx.part{len(parts)}"
        open(os.path.join(EXPORT_DIR, name), "wb").write(chunk)
        parts.append({"name": name, "bytes": len(chunk)})
    return {"model": base_name, "total": len(data), "parts": parts,
            "sha256": hashlib.sha256(data).hexdigest()[:16]}


manifest = {
    "webgpu": split_parts(fp32_path, "ddcolor_webgpu"),
    "wasm": split_parts(fp16_path, "ddcolor_wasm"),
    "inputSize": 512,
}
os.remove(fp32_path)
os.remove(fp16_path)
with open(os.path.join(EXPORT_DIR, "ddcolor.manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print(json.dumps(manifest, indent=2))

zip_path = os.path.join(WORK_DIR, "ddcolor_finetuned")
shutil.make_archive(zip_path, "zip", EXPORT_DIR)
print(f"\n完了！ Drive に保存しました: {zip_path}.zip")
print("→ 展開して public/models/ の同名ファイルに上書きしてください（次のセル参照）")


## 8. サイトへの反映手順

1. Google Drive の `ddcolor_finetune/ddcolor_finetuned.zip` をダウンロードして展開
2. リポジトリの `public/models/` にある以下のファイルを、展開したものに**上書き**
   - `ddcolor_webgpu.onnx.part0` 〜 `partN`
   - `ddcolor_wasm.onnx.part0` 〜 `partN`
   - `ddcolor.manifest.json`
3. **パート数が変わった場合のみ**、`lib/colorization/browser/ortRuntime.ts` の
   `MODELS.ddcolor.files` の配列（part0〜partN のパス一覧）を新しいパート数に合わせて修正
4. `public/models/NOTICE.md` の DDColor 欄に追記:
   「本ノートブックでファインチューニング（日付・学習データ・ステップ数）」
5. `lib/colorization/browser/ortRuntime.ts` の `MODEL_CACHE_NAME` を
   `colorize-model-v2` → `colorize-model-v3` に上げる（会員のブラウザキャッシュを確実に更新）
6. ローカルで `npm run dev` → `/tools/photo-colorize` で古写真をカラー化して確認
7. 問題なければ commit / push（Vercel が自動デプロイ）

### 2回目以降の学習（印刷写真・網点スキャンへの強化）

書籍や新聞に載った写真のスキャン（網点・強コントラスト）が苦手な場合は、
このノートブックの新しい版（網点印刷シミュレーション入り）で**続きから**学習できます:

1. 設定セルで `train_steps` を前回より大きく（例: 6000 → **12000**）
2. あとは同じように上から実行するだけ。Drive のチェックポイント（前回の6000步）
   から自動で続きを学習します
3. 劣化サンプル確認セルで、ドット模様（網点）のかかったサンプルが
   時々出ることを確認してください

### うまくいかないとき
- **色が薄くなった** → `chroma_loss_weight` を 0.3〜0.5 に上げて再学習
- **色が派手すぎ・不自然** → `train_steps` を減らす（学習しすぎ）か `lr_decoder` を 1e-5 に
- **学習前後で変化が小さい** → `dataset: "full"` + `train_steps: 20000` で本格学習
- 元に戻したいときは Git の履歴から以前のモデルファイルを復元すれば即戻せます
